In [ ]:
import sys
sys.path.append('..')

from nodes.writer import write
from config.schemas import Summary, Newsletter

print('Setup complete.')

## Test 1: report_text wird pro Company generiert
Zwei Dummy-Summaries mit key_points. Erwartet: jede Summary hat danach einen nicht-leeren report_text.

In [ ]:
summaries = [
    Summary(
        articles=[],
        company='OpenAI',
        summary_text='OpenAI released GPT-5 with significantly improved reasoning capabilities.',
        key_points=[
            'GPT-5 released on March 2026',
            'Scores 95% on MMLU benchmark',
            'Available via API at $0.01 per 1K tokens',
        ],
    ),
    Summary(
        articles=[],
        company='Anthropic',
        summary_text='Anthropic raised $2B in a new funding round led by Google.',
        key_points=[
            '$2B funding round closed March 2026',
            'Google leads the round, valuation now $30B',
            'Funds will be used for Claude 4 development',
        ],
    ),
]

state = {'summaries': summaries}
result = await write(state)

assert 'summaries' in result, 'Result must contain summaries key'
assert 'newsletter' in result, 'Result must contain newsletter key'

for s in result['summaries']:
    assert len(s.report_text) > 0, f'report_text must not be empty for {s.company}'
    print(f'--- {s.company} ---')
    print(s.report_text)
    print()

print('PASSED')

## Test 2: Newsletter HTML wird korrekt gebaut
Prüft ob das HTML die erwarteten Inhalte enthält.

In [ ]:
newsletter = result['newsletter']

assert isinstance(newsletter, Newsletter), f'Expected Newsletter, got {type(newsletter)}'
assert len(newsletter.html_content) > 0, 'html_content must not be empty'
assert 'Weekly AI Report' in newsletter.html_content, 'HTML must contain title'
assert 'OpenAI' in newsletter.html_content, 'HTML must contain OpenAI'
assert 'Anthropic' in newsletter.html_content, 'HTML must contain Anthropic'
assert 'Executive Summary' in newsletter.html_content, 'HTML must contain Executive Summary section'
assert 'Detailed Report' in newsletter.html_content, 'HTML must contain Detailed Report section'

print('HTML length:', len(newsletter.html_content), 'chars')
print('PASSED')

## HTML-Vorschau

In [ ]:
import os
os.makedirs('preview', exist_ok=True)
with open('preview/newsletter.html', 'w') as f:
    f.write(result['newsletter'].html_content)
print('Saved to tests/preview/newsletter.html')